# Dialforge Sales Acceptance v2.1

Dedicated, cache-proof sales acceptance for Dialforge. This version removes the fragile bootstrap source-rewriting used by v2 and streams the exact child-process output into Colab.

### Run
1. Set **Runtime → Change runtime type → T4 GPU**.
2. Choose **Runtime → Run all**.
3. Keep the tab open until the report appears.


In [ ]:
# ONE-CLICK DIALFORGE SALES ACCEPTANCE v2.1
import base64, json, os, pathlib, subprocess, sys, time, urllib.request
from IPython.display import HTML, FileLink, display
OWNER='SumamaAhmed69'; REPO='Axemetric-Caller-Beta-Runtime'
BOOTSTRAP_PATH='benchmarks/dialforge_sales_colab_bootstrap_v2.py'
BOOTSTRAP=pathlib.Path('/content/dialforge_sales_colab_bootstrap_v2.py')
REPORT=pathlib.Path('/content/dialforge-sales-acceptance/dialforge-sales-acceptance.html')
JSON_REPORT=pathlib.Path('/content/dialforge-sales-acceptance/dialforge-sales-acceptance.json')
ZIP_REPORT=pathlib.Path('/content/dialforge-sales-acceptance-results.zip')
LOG=pathlib.Path('/content/dialforge-sales-v2_1.log')
def api_json(url):
    req=urllib.request.Request(url,headers={'User-Agent':'Dialforge-Sales-v2.1','Accept':'application/vnd.github+json'})
    with urllib.request.urlopen(req,timeout=60) as response: return json.loads(response.read().decode('utf-8'))
print('Dialforge Sales Acceptance launcher: v2.1-dedicated-bootstrap')
print('Resolving one immutable Dialforge source revision...')
source_sha=api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/main?x={time.time_ns()}')['sha']
print('Pinned source:',source_sha)
item=api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/contents/{BOOTSTRAP_PATH}?ref={source_sha}&x={time.time_ns()}')
if item.get('encoding')!='base64' or not item.get('content'): raise RuntimeError('GitHub did not return the dedicated sales bootstrap.')
payload=base64.b64decode(item['content']); text=payload.decode('utf-8')
required=['DIALFORGE SALES COLAB ACCEPTANCE v2.1','dialforge_sales_acceptance.py','base.main()']
for marker in required:
    if marker not in text: raise RuntimeError(f'Dedicated bootstrap marker missing: {marker}')
compile(text,str(BOOTSTRAP),'exec'); BOOTSTRAP.write_bytes(payload)
print(f'Dedicated sales bootstrap verified: {len(payload)} bytes')
env=os.environ.copy(); env['DIALFORGE_SOURCE_SHA']=source_sha
print('\nStarting Dialforge Sales Acceptance v2.1 with LIVE output...\n')
tail=[]
with LOG.open('w',encoding='utf-8') as log:
    proc=subprocess.Popen([sys.executable,str(BOOTSTRAP)],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line,end='')
        log.write(line); log.flush()
        tail.append(line.rstrip('\n'))
        if len(tail)>240: tail=tail[-240:]
    code=proc.wait()
if code!=0:
    print('\n===== LAST 120 LINES =====')
    print('\n'.join(tail[-120:]))
    print('\nFull log:',LOG)
    raise RuntimeError(f'Sales acceptance v2.1 stopped with exit code {code}. The exact failure is shown above.')
if not REPORT.exists() or not JSON_REPORT.exists() or not ZIP_REPORT.exists(): raise RuntimeError('Sales acceptance v2.1 finished without all report files.')
print('\n=== DIALFORGE SALES ACCEPTANCE v2.1 COMPLETE ===')
display(HTML(REPORT.read_text(encoding='utf-8')))
print('\nDownload/share these results:')
display(FileLink(str(ZIP_REPORT))); display(FileLink(str(JSON_REPORT))); display(FileLink(str(LOG)))


## Why v2.1 exists
v2 dynamically rewrote the generic final-acceptance bootstrap. v2.1 uses a dedicated sales bootstrap and captures stdout + stderr line-by-line, so bootstrap failures cannot collapse into a blank RuntimeError.